In [2]:
import pandas as pd
import json
import csv

In [3]:
file_path = "/Users/carolwang/Desktop/yelp_dataset/yelp_academic_dataset_business.json"

# read in business data and convert to df
data = []
with open(file_path, 'r') as f:
    for line in f:
        data.append(json.loads(line))

df = pd.DataFrame(data)

# filter for businesses in CA 
ca_df = df[df['state'] == 'CA']

# filter for Restaurants 
restaurants_ca = ca_df[
    ca_df['categories'].notna() & 
    ca_df['categories'].str.contains('Restaurants')
]

# remove duplicates by keeping the 1st one only (like chains)
restaurants_ca = restaurants_ca.drop_duplicates(subset='name', keep='first')

restaurants_ca = restaurants_ca.reset_index(drop=True)
business_ids = set(restaurants_ca['business_id']) 

print(restaurants_ca.head())
print(restaurants_ca.shape)
restaurants_ca.to_csv("ca_restaurants.csv", index=False)

              business_id                             name  \
0  IDtLPgUrqorrpqSLdfMhZQ             Helena Avenue Bakery   
1  SZU9c8V2GuREDN5KgyHFJw  Santa Barbara Shellfish Company   
2  4xhGQGdGqU60BIznBjqnuA     California Tacos and Taproom   
3  ifjluUv4VASwmFqEp8cWlQ                    Marty's Pizza   
4  VeFfrEZ4iWaecrQg6Eq4cg                         Cal Taco   

                     address           city state postal_code   latitude  \
0      131 Anacapa St, Ste C  Santa Barbara    CA       93101  34.414445   
1          230 Stearns Wharf  Santa Barbara    CA       93101  34.408715   
2  956 Embarcadero Del Norte     Isla Vista    CA       93117  34.411555   
3         2733 De La Vina St  Santa Barbara    CA       93105  34.436236   
4  7320 Hollister Ave, Ste 1         Goleta    CA       93117  34.430542   

    longitude  stars  review_count  is_open  \
0 -119.690672    4.0           389        1   
1 -119.685019    4.0          2404        1   
2 -119.855077    4.0         

In [8]:
reviews_json_path = "/Users/carolwang/Desktop/yelp_dataset/yelp_academic_dataset_review.json"
output_csv_template = "ca_restaurant_reviews_part{}.csv"

fields = ["review_id", "user_id", "business_id", "stars", "date", "text", "useful", "funny", "cool"]

#count total reviews for the businesses
total_reviews = 0
with open(reviews_json_path, 'r') as f:
    for line in f:
        review = json.loads(line)
        if review['business_id'] in business_ids:
            total_reviews += 1

chunk_size = total_reviews // 4  # roughly equal per file

print(f"Total reviews: {total_reviews}, chunk size: {chunk_size}")

# write to 4 separate CSVs
count = 0
file_index = 1
out = open(output_csv_template.format(file_index), 'w', newline='', encoding='utf-8')
writer = csv.DictWriter(out, fieldnames=fields)
writer.writeheader()

with open(reviews_json_path, 'r') as f:
    for line in f:
        review = json.loads(line)
        if review['business_id'] in business_ids:
            writer.writerow(review)
            count += 1
            
            # Switch to next file when reaching chunk_size
            if count >= file_index * chunk_size and file_index < 4:
                out.close()
                file_index += 1
                out = open(output_csv_template.format(file_index), 'w', newline='', encoding='utf-8')
                writer = csv.DictWriter(out, fieldnames=fields)
                writer.writeheader()
                
            if count % 100000 == 0:
                print(f"Wrote {count:,} reviews")

out.close()
print("Done!")

Total reviews: 190483, chunk size: 47620
Wrote 100,000 reviews
Done!


In [11]:
# merge the 2 dataframes with sentiment scores into 1 combined df
csv1 = pd.read_csv("ca_restaurant_reviews_with_sentiment.csv")
csv2 = pd.read_csv("ca_restaurant_reviews_with_sentiment2.csv")
combined = pd.concat([csv1, csv2],ignore_index=True)
combined.to_csv("ca_restaurant_reviews_with_sentiment_combined.csv", index=False)